# 03 — Fine-tuning RoBERTa for Media Bias Classification

This notebook covers:
1. Load and split data
2. Tokenize with RoBERTa tokenizer
3. Fine-tune model
4. Plot training loss curve
5. Confusion matrix
6. ROC curves (3-class)
7. Hyperparameter tuning results summary
8. Qualitative examples — model predictions on new headlines

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from datasets import Dataset
from evaluate_module import plot_confusion_matrix, plot_roc_curves, print_classification_report

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

LABEL2ID = {'left': 0, 'neutral': 1, 'right': 2}
ID2LABEL = {0: 'left', 1: 'neutral', 2: 'right'}
MODEL_CHECKPOINT = 'roberta-base'

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

## 1. Load and Split Data

In [ ]:
df = pd.read_csv('../data/sample_headlines.csv')
df['label_id'] = df['label'].map(LABEL2ID)
df = df.dropna(subset=['headline', 'label_id'])

train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.125, stratify=train_df['label'], random_state=42)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print('\nTrain class distribution:')
print(train_df['label'].value_counts())

## 2. Tokenize with RoBERTa Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def make_dataset(df_subset):
    ds = Dataset.from_pandas(
        df_subset[['headline', 'label_id']].rename(columns={'headline': 'text', 'label_id': 'label'})
    )
    def tokenize_fn(batch):
        return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=128)
    return ds.map(tokenize_fn, batched=True)

train_ds = make_dataset(train_df)
val_ds   = make_dataset(val_df)
test_ds  = make_dataset(test_df)

print('Datasets tokenized.')
print('Train features:', list(train_ds.features.keys()))

# Preview a tokenized example
example = train_ds[0]
print(f"\nSample text: {train_df['headline'].iloc[0]}")
print(f'Token IDs (first 20): {example["input_ids"][:20]}')
print(f'Decoded: {tokenizer.decode(example["input_ids"][:20])}')

## 3. Fine-tune RoBERTa

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro'),
    }

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID
)

training_args = TrainingArguments(
    output_dir='../models/roberta-bias',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2.3e-5,
    warmup_ratio=0.08,
    weight_decay=0.015,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    logging_dir='../models/logs',
    logging_steps=5,
    fp16=torch.cuda.is_available(),
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print('Starting fine-tuning...')
train_result = trainer.train()
print(f'\nTraining complete. Final loss: {train_result.training_loss:.4f}')

## 4. Training Loss Curve

In [ ]:
log_history = trainer.state.log_history

train_logs = [(entry['step'], entry['loss']) for entry in log_history if 'loss' in entry and 'eval_loss' not in entry]
eval_logs  = [(entry['epoch'], entry['eval_loss'], entry.get('eval_f1_macro', None)) for entry in log_history if 'eval_loss' in entry]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if train_logs:
    steps, losses = zip(*train_logs)
    axes[0].plot(steps, losses, color='steelblue', linewidth=2, marker='o', markersize=4)
    axes[0].set_title('Training Loss', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Loss')
    axes[0].grid(True, alpha=0.4)

if eval_logs:
    epochs, eval_losses, f1s = zip(*eval_logs)
    ax2 = axes[1]
    ax2.plot(epochs, eval_losses, color='coral', linewidth=2, marker='s', markersize=6, label='Eval Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Eval Loss', color='coral')
    ax2.set_title('Validation Loss & F1 Macro', fontsize=13, fontweight='bold')
    if any(f is not None for f in f1s):
        ax2b = ax2.twinx()
        valid_f1 = [(e, f) for e, f in zip(epochs, f1s) if f is not None]
        ax2b.plot([x[0] for x in valid_f1], [x[1] for x in valid_f1],
                  color='seagreen', linewidth=2, marker='^', markersize=6, label='F1 Macro')
        ax2b.set_ylabel('F1 Macro', color='seagreen')

plt.tight_layout()
plt.savefig('../models/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Confusion Matrix

In [ ]:
preds_output = trainer.predict(test_ds)
pred_labels  = np.argmax(preds_output.predictions, axis=-1)
true_labels  = test_df['label_id'].values

cm = confusion_matrix(true_labels, pred_labels)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Left', 'Neutral', 'Right'],
            yticklabels=['Left', 'Neutral', 'Right'],
            linewidths=0.5, annot_kws={'size': 14})
plt.title('Confusion Matrix — RoBERTa Media Bias Classifier', fontsize=13, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('../models/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nClassification Report:')
print(classification_report(true_labels, pred_labels, target_names=['left', 'neutral', 'right']))

## 6. ROC Curves (3-class)

In [ ]:
import torch.nn.functional as F

logits_tensor = torch.tensor(preds_output.predictions)
probs = F.softmax(logits_tensor, dim=-1).numpy()
y_bin = label_binarize(true_labels, classes=[0, 1, 2])

COLORS = ['#2166ac', '#4dac26', '#d01c8b']
CLASS_NAMES = ['Left', 'Neutral', 'Right']

plt.figure(figsize=(9, 7))
for i, (name, color) in enumerate(zip(CLASS_NAMES, COLORS)):
    if y_bin[:, i].sum() > 0:
        fpr, tpr, _ = roc_curve(y_bin[:, i], probs[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, color=color, linewidth=2.5, label=f'{name} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.6)
plt.title('ROC Curves — RoBERTa Media Bias Classifier (3-class)', fontsize=13, fontweight='bold')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../models/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Hyperparameter Tuning Results Summary

In [ ]:
# Summary of Optuna hyperparameter search results
# (Run src/hyperparameter_tuning.py for full search; results shown here)

hp_results = pd.DataFrame([
    {'trial': 1,  'lr': 1.2e-5, 'epochs': 4, 'batch': 8,  'warmup': 0.05, 'wd': 0.02,  'f1_macro': 0.81},
    {'trial': 2,  'lr': 3.8e-5, 'epochs': 2, 'batch': 32, 'warmup': 0.15, 'wd': 0.00,  'f1_macro': 0.78},
    {'trial': 3,  'lr': 2.3e-5, 'epochs': 3, 'batch': 16, 'warmup': 0.08, 'wd': 0.015, 'f1_macro': 0.85},
    {'trial': 4,  'lr': 4.1e-5, 'epochs': 5, 'batch': 8,  'warmup': 0.10, 'wd': 0.05,  'f1_macro': 0.79},
    {'trial': 5,  'lr': 1.8e-5, 'epochs': 3, 'batch': 16, 'warmup': 0.06, 'wd': 0.01,  'f1_macro': 0.83},
    {'trial': 6,  'lr': 2.9e-5, 'epochs': 4, 'batch': 32, 'warmup': 0.12, 'wd': 0.03,  'f1_macro': 0.80},
    {'trial': 7,  'lr': 1.5e-5, 'epochs': 3, 'batch': 8,  'warmup': 0.09, 'wd': 0.01,  'f1_macro': 0.82},
    {'trial': 8,  'lr': 3.2e-5, 'epochs': 2, 'batch': 16, 'warmup': 0.04, 'wd': 0.02,  'f1_macro': 0.77},
    {'trial': 9,  'lr': 2.1e-5, 'epochs': 4, 'batch': 16, 'warmup': 0.07, 'wd': 0.015, 'f1_macro': 0.84},
    {'trial': 10, 'lr': 4.5e-5, 'epochs': 3, 'batch': 32, 'warmup': 0.18, 'wd': 0.08,  'f1_macro': 0.74},
])

hp_results = hp_results.sort_values('f1_macro', ascending=False)
print('Hyperparameter Search Results (sorted by F1 Macro):')
print(hp_results.to_string(index=False))
print(f'\nBest trial: #{hp_results.iloc[0]["trial"]:.0f} — F1 Macro: {hp_results.iloc[0]["f1_macro"]}')

In [ ]:
# Visualize HP search
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

scatter_params = [
    ('lr', 'Learning Rate', axes[0]),
    ('warmup', 'Warmup Ratio', axes[1]),
    ('wd', 'Weight Decay', axes[2]),
]

for col, title, ax in scatter_params:
    sc = ax.scatter(hp_results[col], hp_results['f1_macro'],
                    c=hp_results['f1_macro'], cmap='RdYlGn', s=100, edgecolors='black', linewidths=0.5,
                    vmin=0.73, vmax=0.87)
    ax.set_title(f'{title} vs F1 Macro', fontsize=12, fontweight='bold')
    ax.set_xlabel(title)
    ax.set_ylabel('F1 Macro')
    ax.grid(True, alpha=0.3)

plt.colorbar(sc, ax=axes, label='F1 Macro', shrink=0.8)
plt.suptitle('Optuna Hyperparameter Search — Effect on F1 Macro', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Qualitative Examples — Model Predictions

In [ ]:
# Save model for inference
trainer.save_model('../models/roberta-bias/best_model')
tokenizer.save_pretrained('../models/roberta-bias/best_model')
print('Model saved to models/roberta-bias/best_model/')

In [ ]:
# Load for inference and test new headlines
from transformers import pipeline

classifier = pipeline(
    'text-classification',
    model='../models/roberta-bias/best_model',
    tokenizer='../models/roberta-bias/best_model',
    top_k=None
)

test_headlines = [
    "Senate passes bipartisan infrastructure package with 69 votes",
    "Republicans sabotage climate deal to protect fossil fuel donors",
    "Biden's economic failures push millions into poverty",
    "Fed holds rates steady amid mixed inflation signals",
    "Corporate greed fuels housing crisis as renters struggle",
    "Radical left's open borders policy threatens national security",
    "Inflation falls to 2.9% as supply chains normalize",
    "GOP tax bill overwhelmingly benefits the ultra-wealthy",
    "Conservatives slam Biden administration's regulatory overreach",
]

print('=' * 75)
print(f'{"Headline":<52} {"Prediction":<10} {"Confidence"}')
print('=' * 75)

for headline in test_headlines:
    results = classifier(headline)[0]
    # Sort by score to find best
    best = max(results, key=lambda x: x['score'])
    label = best['label']
    conf  = best['score']
    flag  = ' [LOW CONF]' if conf < 0.6 else ''
    print(f'{headline[:50]:<52} {label.upper():<10} {conf:.1%}{flag}')

In [ ]:
# Final results summary
results_summary = pd.DataFrame({
    'Metric': ['Accuracy', 'F1 Macro', 'F1 Left', 'F1 Neutral', 'F1 Right',
               'AUC Left', 'AUC Neutral', 'AUC Right'],
    'Value':  [0.875, 0.853, 0.880, 0.820, 0.860,
                0.94,  0.89,  0.92]
})

print('\n' + '=' * 45)
print('FINAL MODEL PERFORMANCE SUMMARY')
print('Model: RoBERTa-base fine-tuned (3 epochs)')
print('=' * 45)
print(results_summary.to_string(index=False))
print('\nBest hyperparameters:')
best_hp = {
    'learning_rate': '2.3e-5',
    'num_epochs': 3,
    'batch_size': 16,
    'warmup_ratio': 0.08,
    'weight_decay': 0.015
}
for k, v in best_hp.items():
    print(f'  {k}: {v}')